## *We already know about Query, Key and Values in previous notebook. Here we are learning about sccore functions.*

# Score Function in Attention Mechanism


In attention mechanisms, the **score function** (also called the **similarity function**) measures how well each *key* matches the *query*.  
- The score determines **how much focus (weight)** the model should place on each value when computing the attention output.  
- In other words, it is a **relevance measure** between query `q` and key `k`.


<div align='center'>

[![scoring-function.png](https://i.postimg.cc/jjVDkZ3t/scoring-function.png)](https://postimg.cc/PC4fCzkR)

*fig:attention scoring function*

</div>

The image illustrates **how score functions operate** in attention:

- On the **left**, we have a **Query (Q)** represented as a single vector.  
- On the **right**, we have multiple **Keys (K)**, each paired with a **Value (V)**.  
- The query is compared with each key through the **score function**.  
- These scores (similarities) are then **normalized** (typically with Softmax) to produce **attention weights**.  
- Finally, these weights are used to compute a **weighted sum of the values**, producing the **context vector** (the final output of attention).


This figure shows the **core step of attention pooling**: using a score function to link queries with keys, and then aggregating values accordingly.


## General Mathematical Formulation
Given:
- Query vector: **q**
- Key vector: **k**
- Score function: **f(q, k)**

The **attention weight** is computed as:

$
\alpha_i = \frac{\exp(f(q, k_i))}{\sum_j \exp(f(q, k_j))}
$

Then the **attention output** is:

$
\text{Attention}(q, K, V) = \sum_i \alpha_i v_i
$

Where:
- \$( k_i \)$ → keys  

- \$( v_i \)$ → values

- \$( \alpha_i \)$ → normalized attention weights  






# Attention Mechanisms: Dot-Product, Additive, and Multiplicative Score Function

## **Overview**

Attention mechanisms allow neural networks to focus on specific parts of the input when making predictions. Different attention mechanisms use different **scoring functions** to compute attention weights. This notebook covers three fundamental attention mechanisms:

1. **Dot-Product Attention** (Scaled and Unscaled)
2. **Additive Attention** (Bahdanau Attention)
3. **Multiplicative Attention** (Luong Attention)


## 1. Dot-Product Attention

### Unscaled Dot-Product Attention  

Unscaled dot-product attention is the simplest form of attention where the similarity between a query and a key is measured using their dot product. The resulting score indicates how much focus a query should place on a given key. Since it does not apply any scaling, the values can grow large, causing the softmax function to produce very sharp distributions. This method is effective for **small-dimensional embeddings**, but not ideal for high dimensions due to instability.  

### Mathematical Formulation  

$
\alpha_{ij} = \text{softmax}(Q_i \cdot K_j^{T})
$

where  
- \$( Q_i \)$ = query vector for token *i*  

- $( K_j)$ = key vector for token *j*  
- $(\alpha_{ij})$ = attention weight between query *i* and key *j*  

The output is then:  

$
\text{Attention}(Q, K, V) = \sum_j \alpha_{ij} V_j
$


###  When to Use  
- Best for **small embedding dimensions** where dot products remain stable.  
- For **large dimensions**, prefer **scaled dot-product attention** to avoid very large values and unstable gradients.  


### **Scaled Dot-Product Attention**

When the query and key vectors share the same dimension \(d\), **dot-product attention** is computationally efficient and widely used—especially in Transformers.



##  Mathematical Formulation

For query matrix $(Q \in \mathbb{R}^{n_q \times d_k})$, key matrix $(K \in \mathbb{R}^{n_k \times d_k})$, and value matrix $(V \in \mathbb{R}^{n_k \times d_v})$, the **scaled dot-product attention** is defined as:

$
\text{Attention}(Q, K, V) = \mathrm{softmax}\left(\frac{Q K^\top}{\sqrt{d_k}}\right) V
$

Here:
- \$(Q K^\top\)$ produces a **score matrix** of shape

 > > \$(n_q \times n_k\)$  

- Each entry (i,j) is the dot product between query (q_i\) and key (k_j)
- We scale by $(\sqrt{d_k})$ to stabilize gradients  
- Apply softmax row-wise → obtain attention weights  
- Finally, compute the **weighted sum** over \(V\) → output shape \$(n_q \times d_v\)$


##  Why $(Q K^\top)$ — Not $(Q^\top K)?$

- **$(Q K^\top)$** results in an **$(n_q \times n_k)$ matrix**, where each row corresponds to one query's similarity to all keys. This aligns perfectly for:
$
  \text{Attention}(Q, K, V) = \text{softmax scores} \times V
$

- **$(Q^\top K)$** yields a **$(d_k \times d_k)$** matrix (an inner product across all feature dimensions), which doesn't align with values \$(V\)$ for weighted summation.

Hence, \$(Q K^\top\)$ is the correct form to compute attention weights across query–key pairs.






### Advantages
- **Computationally Efficient**: Simple matrix operations
- **Parallelizable**: All attention scores computed simultaneously
- **Hardware Optimized**: Matrix multiplication is highly optimized on GPUs

### Disadvantages
- **Requires Same Dimensions**: Query and key must have same dimensionality
- **May Suffer from Large Values**: Without scaling, softmax can saturate

### When to Use
- **Transformers**: Primary attention mechanism in BERT, GPT, etc.
- **Self-Attention**: When Q, K, V come from the same sequence
- **Large-Scale Models**: When computational efficiency is crucial
- **When d_k is moderate to large**: Scaling becomes important for larger dimensions


### **2. Additive Attention (Bahdanau Attention)**  

Additive attention, introduced by Bahdanau et al. (2014), computes attention scores using a feed-forward neural network instead of a dot product. The query and key vectors are combined, passed through a hidden layer with a non-linear activation (usually `tanh`), and then scored with a learnable weight vector. This allows the model to **learn a richer similarity function** than plain dot products, making it more flexible, though computationally more expensive.

<div align='center'>

[![bhadanau1.png](https://i.postimg.cc/VL58FNRw/bhadanau1.png)](https://postimg.cc/ppN4WR4c)

[![bahdanau-attention.jpg](https://i.postimg.cc/bNP6zVZB/bahdanau-attention.jpg)](https://postimg.cc/kDjNscFQ)

*figure: Bahdanhau Attention Mechanism*

</div>

1. Producing the Encoder Hidden States - Encoder produces hidden states of each element in the input sequence

2. ### Mathematical Formulation  

>> The attention score between query \( Q_i \) and key \( K_j \) is:  

$$
e_{ij} = v^T \tanh(W_q Q_i + W_k K_j)
$$

> where  
- $( W_q)$ and $( W_k)$ = learnable weight matrices  
- \( v \) = learnable weight vector  
- $( e_{ij}$) = alignment score between query *i* and key *j*  

> The attention weights are then:  

$$
\alpha_{ij} = \text{softmax}(e_{ij})
$$

> The final context vector is:  

$$
\text{Attention}(Q, K, V) = \sum_j \alpha_{ij} V_j
$$

 $$ Context Vector = Encoder Output = Attention Wights $$

3. Decoding the Output - the context vector is concatenated with the previous decoder output and fed into the Decoder RNN for that time step along with the previous decoder hidden state to produce a new output

4. The process (steps 2-5) repeats itself for each time step of the decoder until an token is produced or output is past the specified maximum length


### When to Use  
- Suitable when we want **more expressive attention scoring** beyond dot products.  
- Often used in **sequence-to-sequence models** (like machine translation with RNNs).  
- More computationally expensive than dot-product attention but can perform better on **small to medium dimensions**.  


### Advantages
- **Flexible Dimensions**: Query and key can have different dimensions
- **More Expressive**: Non-linear transformation allows complex attention patterns
- **Learnable Complexity**: Attention dimension is a hyperparameter

### Disadvantages
- **More Parameters**: Requires additional weight matrices
- **Sequential Computation**: Less parallelizable than dot-product
- **Computational Overhead**: More operations per attention computation


## **3. Multiplicative Attention (Luong Attention)**


Multiplicative attention, proposed by Luong et al. (2015), is a more efficient variant of Bahdanau attention. Instead of using a feed-forward network, it computes attention scores using a dot product between the query and key vectors, sometimes with an additional learnable weight matrix. This makes it computationally cheaper than additive attention, while still effective.  

<div align= 'center'>

[![Luong-Attention.jpg](https://i.postimg.cc/7LjVjKvX/Luong-Attention.jpg)](https://postimg.cc/gwq8Zq26)

</div>


The entire step-by-step process of applying Attention in Luong’s paper is as follows:

1. Producing the Encoder Hidden States - Encoder produces hidden states of each element in the input sequence


2. Decoder RNN - the previous decoder hidden state and decoder output is passed through the Decoder RNN to generate a new hidden state for that time step

3. ### Mathematical Formulation  

>> The alignment score between query \( Q_i \) and key \( K_j \) can be defined in three forms:  

1. **Dot:**  
$
e_{ij} = Q_i \cdot K_j^T
$

2. **General:**  
$
e_{ij} = Q_i W_a K_j^T
$

3. **Concat (less common in Luong’s version):**  
$
e_{ij} = v^T \tanh(W [Q_i ; K_j])
$

>> where  
- $( W_a $) = learnable weight matrix (for the general form)  
- \$( e_{ij} \)$ = alignment score  

>> The attention weights are:  

$$
\alpha_{ij} = \text{softmax}(e_{ij})
$$

The context vector is:  

$$
\text{Attention}(Q, K, V) = \sum_j \alpha_{ij} V_j
$$



4. Softmaxing the Alignment Scores - the alignment scores for each encoder hidden state are combined and represented in a single vector and subsequently softmaxed


5. Calculating the Context Vector - the encoder hidden states and their respective alignment scores are multiplied to form the context vector


6. Producing the Final Output - the context vector is concatenated with the decoder hidden state generated in step 2 as passed through a fully connected layer to produce a new output


7. The process (steps 2-6) repeats itself for each time step of the decoder until an token is produced or output is past the specified maximum length





### When to Use  
- Useful when efficiency is important, since it avoids the heavy computation of additive attention.  
- Works well with **larger embedding dimensions**.  
- Commonly used in **machine translation** models and is a precursor to the attention mechanism used in Transformers.  
